# ML Assignment 2 - Adult Income Classification

Running the full assignment here: dataset check, preprocessing, the 5 models and the metrics.
All the actual code is in the .py files in this folder; the notebook just calls them step by step.

In [ ]:
import os, sys
# so that imports work from model/ or from the project root
HERE = os.path.abspath("model") if os.path.isdir("model") else os.path.abspath(".")
sys.path.insert(0, HERE)

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display

import data_preprocessing as dp
import logistic_regression, decision_tree, knn, naive_bayes, random_forest
from evaluation import metrics_table
import train_all
print("Project folder:", dp.PROJECT_DIR)

## Step 1 - Load the dataset

In [ ]:
df = dp.load_raw_data()
print("Shape:", df.shape)
df.head()

In [ ]:
s = dp.dataset_summary(df)
print("Instances :", s["n_instances"])
print("Features  :", s["n_features"], f"({len(s['numeric_features'])} numeric, {len(s['categorical_features'])} categorical)")
print("Numeric   :", s["numeric_features"])
print("Categorical:", s["categorical_features"])
print("Target    : income  (<=50K / >50K)")
df.info()

### Missing values, duplicates and class balance

In [ ]:
print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nRows with at least one missing value:", s["rows_with_missing"])
print("Exact duplicate rows:", s["n_duplicates"])
print("\nClass distribution:")
print(pd.DataFrame({"count": df["income"].value_counts(), "percent": df["income"].value_counts(normalize=True).mul(100).round(2)}))
print("\nImbalance ratio (majority : minority) = %.2f : 1" % (s["class_counts"]["<=50K"] / s["class_counts"][">50K"]))

## Step 2 - Preprocessing and train/test split

In [ ]:
X, y, n_dupes = dp.clean_data(df)          # remove duplicates, drop education, target to 0/1
X_train, X_test, y_train, y_test = dp.split_data(X, y)   # 80:20 stratified, seed 42
print("Duplicates removed:", n_dupes, "-> rows:", len(X))
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Share of >50K  -> train: %.4f | test: %.4f  (class distribution maintained)" % (y_train.mean(), y_test.mean()))
dp.save_test_data(X_test, y_test)
print("test_data.csv saved")

In [ ]:
# preprocessing (impute + one hot + scale) is part of each model's pipeline and is fit on train only
# just checking the shape after encoding:
prep = dp.build_preprocessor(scale_numeric=True)
Xt = prep.fit(X_train).transform(X_train)
print("Encoded training matrix shape:", Xt.shape)
prep

## Step 3 - Train and evaluate the 5 models
Uses the best params from my grid search. Set TUNE = True to run the grid search again (slow, ~10-15 min).

In [ ]:
TUNE = False
results = []
for module in [logistic_regression, decision_tree, knn, naive_bayes, random_forest]:
    results.append(module.run(X_train, X_test, y_train, y_test, tune=TUNE))

## Comparison table

In [ ]:
table = metrics_table(results).round(4)
table

## Graphs

In [ ]:
os.makedirs(train_all.PLOTS_DIR, exist_ok=True)
train_all.plot_class_distribution(df)
train_all.plot_missing_values(df)
train_all.plot_metric_comparison(table)
train_all.plot_metric_heatmap(table)
train_all.plot_roc_curves(results, y_test)
train_all.plot_confusion_matrices(results, y_test)
for f in ["class_distribution.png", "missing_values.png", "metrics_comparison_bar.png",
          "metrics_heatmap.png", "roc_curves.png", "confusion_matrices.png"]:
    display(Image(os.path.join(train_all.PLOTS_DIR, f)))

## Which model is best for each metric

In [ ]:
print(pd.DataFrame({"Best model": table.idxmax(), "Score": table.max()}))